# 🧠 Neuro-Spicy Support Assistant: Notebook Edition

A Jupyter version of the Streamlit app. It covers the same features: capacity tracking, medications, meals, activity, the calendar, favorite foods, the shopping list, and the local-LLM Decision Assistant.

**It shares data with the Streamlit app.** Both read and write the same JSON files in `neuro-spicy-app/`, so anything you log here shows up in the app and the other way round.

### How to run
1. From `neuro-spicy-app/`: `pip install -r requirements.txt -r requirements-notebook.txt`
2. Start Jupyter **from `neuro-spicy-app/`** (`jupyter lab`) and open this notebook.
3. Run all cells (**Run ▸ Run All Cells**). Each section shows interactive widgets, and the final cell puts them all in one tabbed dashboard.
4. The Decision Assistant needs Ollama running with the `llama3.2-3b-ft` model (see `setup_model.sh`).

Set the `OLLAMA_URL` / `OLLAMA_MODEL` environment variables to use a different endpoint or model.

## 1. Setup

In [ ]:
import html
import json
import os
import re
import sys
import uuid
from datetime import datetime, time as dtime
from pathlib import Path

import ipywidgets as w
import pandas as pd
import requests
from IPython.display import HTML, Markdown, display

# The app's modules use paths relative to the working directory, so the notebook
# must run from neuro-spicy-app/ to share the same data files.
APP_DIR = Path.cwd()
if not (APP_DIR / "app.py").exists() and (APP_DIR / "neuro-spicy-app" / "app.py").exists():
    APP_DIR = APP_DIR / "neuro-spicy-app"
    os.chdir(APP_DIR)
assert (APP_DIR / "app.py").exists(), "Start Jupyter from the neuro-spicy-app/ directory."
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

# Reuse the Streamlit app's data layer. These modules import streamlit, which works
# outside a Streamlit server; silence its "missing ScriptRunContext" warnings.
import logging
logging.getLogger("streamlit.runtime.scriptrunner_utils.script_run_context").addFilter(
    lambda record: "missing ScriptRunContext" not in record.getMessage()
)

import activity_tracker as act
import calendar_component as cal
import meal_tracker as meal
import medication_tracker as med
import safe_food_library as sfl
from json_store import save_json

print(f"Data directory: {APP_DIR}")

In [ ]:
# --- Shared helpers -----------------------------------------------------------

CAPACITY_STATE_FILE = "capacity_state.json"
SETTINGS_FILE = "widget_settings.json"
MAX_SPOONS = 12
MAX_CUPS = 16


def today_str():
    return datetime.now().strftime("%Y-%m-%d")


def load_capacity():
    """Today's capacity counters. Same file format as the Streamlit app; resets daily."""
    default = {"spoons_count": MAX_SPOONS, "water_count": 0, "synced_drink_count": 0, "date": today_str()}
    try:
        data = json.loads(Path(CAPACITY_STATE_FILE).read_text())
    except (FileNotFoundError, ValueError):
        return default
    if data.get("date", today_str()) != today_str():
        return default
    for key, value in default.items():
        data.setdefault(key, value)
    return data


def update_capacity(**changes):
    """Read-modify-write against the file so we never write back stale values."""
    state = load_capacity()
    state.update(changes)
    state["date"] = today_str()
    save_json(CAPACITY_STATE_FILE, state)
    return state


def sync_hydration():
    """Mirrors app.py: each Drink logged today adds one cup to the water count."""
    drinks = sum(
        1 for m in meal.load_meal_log()
        if m.get("type") == "Drink" and m.get("date_str") == today_str()
    )
    state = load_capacity()
    new = drinks - state["synced_drink_count"]
    if new > 0:
        state = update_capacity(water_count=min(MAX_CUPS, state["water_count"] + new), synced_drink_count=drinks)
    elif drinks != state["synced_drink_count"]:
        state = update_capacity(synced_drink_count=drinks)
    return state


def energy_from_spoons(spoons):
    if spoons <= 1:
        return "None"
    if spoons <= 5:
        return "Low"
    if spoons <= 9:
        return "Medium"
    return "High"


def load_user_name():
    try:
        return json.loads(Path(SETTINGS_FILE).read_text()).get("user_name", "").strip()
    except (FileNotFoundError, ValueError):
        return ""


def esc(value):
    return html.escape(str(value if value is not None else ""))


# Sections register a refresh callback so that logging something in one section
# (e.g. a drink) updates the others (e.g. the hydration counter and calendar).
_refreshers = []


def register(refresh_fn):
    _refreshers.append(refresh_fn)
    refresh_fn()
    return refresh_fn


def refresh_all():
    cal.sync_daily_logs_to_calendar()
    for fn in list(_refreshers):
        fn()


def status_label():
    return w.HTML(layout=w.Layout(min_height="1.5em"))


def flash(label, message, ok=True):
    color = "#15803d" if ok else "#b91c1c"
    label.value = f"<span style='color:{color}'>{esc(message)}</span>"


def table_html(rows, headers):
    if not rows:
        return "<i>Nothing here yet.</i>"
    head = "".join(f"<th style='text-align:left;padding:4px 10px'>{esc(h)}</th>" for h in headers)
    body = "".join(
        "<tr>" + "".join(f"<td style='padding:4px 10px'>{cell}</td>" for cell in row) + "</tr>"
        for row in rows
    )
    return f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"

## 2. Daily Capacity Dashboard
Spoons (energy) and hydration for today, plus the wake/sleep buttons. Values are saved as you change them. Drinks logged in the Meals section count toward hydration automatically.

In [ ]:
def build_capacity_section():
    title = w.HTML()
    spoons = w.IntSlider(min=0, max=MAX_SPOONS, description="🥄 Spoons", continuous_update=False,
                         layout=w.Layout(width="420px"))
    water = w.IntSlider(min=0, max=MAX_CUPS, description="🥛 Cups", continuous_update=False,
                        layout=w.Layout(width="420px"))
    energy = w.HTML()
    status = status_label()
    loading = {"active": False}

    def refresh():
        loading["active"] = True
        state = sync_hydration()
        spoons.value = state["spoons_count"]
        water.value = state["water_count"]
        name = load_user_name()
        heading = f"{name}' Daily Dashboard" if name.endswith("s") else f"{name}'s Daily Dashboard"
        title.value = f"<h2 style='margin:0'>{esc(heading if name else 'Daily Capacity & Dashboard')}</h2>"
        energy.value = (f"Energy level: <b>{energy_from_spoons(spoons.value)}</b> · "
                        f"Hydration: <b>{water.value} / {MAX_CUPS}</b> cups")
        loading["active"] = False

    def on_spoons(change):
        if not loading["active"]:
            update_capacity(spoons_count=change["new"])
            refresh()

    def on_water(change):
        if not loading["active"]:
            update_capacity(water_count=change["new"])
            refresh()

    spoons.observe(on_spoons, names="value")
    water.observe(on_water, names="value")

    def quick_water(_):
        meal.log_meal_entry("Glass of Water", meal_type="Drink")
        flash(status, "Logged 1 cup of water!")
        refresh_all()

    def wake_sleep(event_name):
        def handler(_):
            logged = act.log_wake_sleep_event(event_name)
            flash(status, f"{event_name} logged at {logged.strftime('%I:%M %p')}")
            refresh_all()
        return handler

    btn_water = w.Button(description="🥛 Quick Log Water")
    btn_wake = w.Button(description="☀ I'm awake!")
    btn_sleep = w.Button(description="🌙 Sleep time!")
    btn_reload = w.Button(description="🔄 Reload", tooltip="Re-read data changed by the Streamlit app")
    btn_water.on_click(quick_water)
    btn_wake.on_click(wake_sleep("Woke Up"))
    btn_sleep.on_click(wake_sleep("Went to Sleep"))
    btn_reload.on_click(lambda _: refresh_all())

    register(refresh)
    return w.VBox([title, spoons, water, energy,
                   w.HBox([btn_water, btn_wake, btn_sleep, btn_reload]), status])


display(build_capacity_section())

## 3. 💊 Medications
Mark today's doses as taken. A dose only counts on the day it was logged.

In [ ]:
def build_medication_section():
    rows_box = w.VBox()
    status = status_label()

    def refresh():
        meds = med.load_medications()
        rows = []
        for idx, m in enumerate(meds):
            taken = med.is_taken_today(m)
            label = w.HTML(
                f"<b>{esc(m.get('name'))}</b> (<code>{esc(m.get('dosage') or 'N/A')}</code>) · "
                f"scheduled {esc(m.get('time', 'N/A'))} · "
                + (f"<span style='color:#15803d'>✅ Taken at {esc(m.get('logged_at_time'))}</span>"
                   if taken else "<span style='color:#64748b'>Pending</span>"),
                layout=w.Layout(width="520px"),
            )
            btn = w.Button(description="Reset" if taken else "Mark Taken",
                           button_style="" if taken else "success", layout=w.Layout(width="120px"))
            btn.on_click(make_toggle(idx, taken, m.get("name")))
            rows.append(w.HBox([label, btn]))
        rows_box.children = rows or [w.HTML("<i>No medications added yet.</i>")]

    def make_toggle(idx, taken, name):
        def handler(_):
            if taken:
                meds = med.load_medications()
                if 0 <= idx < len(meds):
                    meds[idx]["taken"] = False
                    meds[idx]["logged_at_time"] = None
                    med.save_medications(meds)
                flash(status, f"Reset {name}.")
            else:
                med.log_medication_taken(idx)
                flash(status, f"Logged {name} as taken!")
            refresh_all()
        return handler

    name = w.Text(placeholder="e.g. Advil, Vitamin D", description="Name")
    dosage = w.Text(placeholder="e.g. 200mg, 1 tablet", description="Dosage")
    sched = w.TimePicker(value=datetime.now().time().replace(second=0, microsecond=0), description="Time")
    add = w.Button(description="➕ Add Medication", button_style="primary")

    def on_add(_):
        if not name.value.strip():
            flash(status, "Please enter a medication name.", ok=False)
            return
        meds = med.load_medications()
        when = sched.value or datetime.now().time()
        meds.append({
            "name": name.value.strip(),
            "dosage": dosage.value.strip(),
            "time": when.strftime("%I:%M %p").lstrip("0"),
            "taken": False,
            "logged_at_time": None,
        })
        med.save_medications(meds)
        flash(status, f"Added {name.value.strip()} to schedule!")
        name.value = dosage.value = ""
        refresh_all()

    add.on_click(on_add)
    register(refresh)
    return w.VBox([rows_box, w.HTML("<b>Add a medication</b>"), w.HBox([name, dosage]),
                   w.HBox([sched, add]), status])


display(build_medication_section())

## 4. 🍽️ Meals
Log meals, snacks, and drinks. Drinks also count toward hydration.

In [ ]:
def build_meal_section():
    listing = w.HTML()
    status = status_label()

    def refresh():
        today = [m for m in meal.load_meal_log()
                 if m.get("date_str") == today_str() or str(m.get("timestamp", "")).startswith(today_str())]
        listing.value = table_html(
            [[esc(m.get("time_str", m.get("time", ""))), f"<b>{esc(m.get('name', m.get('item', 'Meal')))}</b>",
              esc(m.get("type", "Meal")), f"{esc(m.get('calories', 0))} kcal", esc(m.get("effort", "N/A"))]
             for m in reversed(today)],
            ["Time", "Item", "Type", "Calories", "Prep effort"],
        )

    name = w.Text(placeholder="e.g. Oatmeal with berries", description="Food")
    kind = w.Dropdown(options=["Breakfast", "Lunch", "Dinner", "Snack", "Drink"], description="Type")
    calories = w.BoundedIntText(value=300, min=0, max=3000, step=50, description="Calories")
    effort = w.Dropdown(options=["None", "Low", "Medium", "High"], description="Prep effort")
    add = w.Button(description="Log Meal", button_style="primary")

    def on_add(_):
        if not name.value.strip():
            flash(status, "Please enter a meal or food item.", ok=False)
            return
        meal.log_meal_entry(name.value.strip(), meal_type=kind.value,
                            calories=calories.value, effort=effort.value)
        flash(status, f"Logged {name.value.strip()}!")
        name.value = ""
        refresh_all()

    add.on_click(on_add)
    register(refresh)
    return w.VBox([w.HTML("<b>Today's meals</b>"), listing, w.HBox([name, kind]),
                   w.HBox([calories, effort, add]), status])


display(build_meal_section())

## 5. 🧘 Activity
Log movement along with its intensity and sensory environment.

In [ ]:
def build_activity_section():
    listing = w.HTML()
    status = status_label()

    def refresh():
        today = [a for a in act.load_activity_log()
                 if a.get("date_str") == today_str() or str(a.get("timestamp", "")).startswith(today_str())]
        listing.value = table_html(
            [[esc(a.get("time", a.get("time_str", ""))), f"<b>{esc(a.get('name', 'Activity'))}</b>",
              f"{esc(a['duration'])} mins" if a.get("duration") else "",
              esc(a.get("intensity", "")), esc(a.get("sensory", ""))]
             for a in reversed(today)],
            ["Time", "Activity", "Duration", "Intensity", "Environment"],
        )

    name = w.Text(placeholder="e.g. Walking, Stretches", description="Activity")
    duration = w.BoundedIntText(value=15, min=1, max=300, step=5, description="Minutes")
    intensity = w.Dropdown(options=["Gentle / Low Demand", "Moderate", "High Energy"], description="Intensity")
    sensory = w.Dropdown(options=["Calm & Quiet", "Outdoors / Nature", "Music / Upbeat", "High Stimulus"],
                         description="Sensory")
    add = w.Button(description="Log Activity", button_style="primary")

    def on_add(_):
        if not name.value.strip():
            flash(status, "Please enter an activity name.", ok=False)
            return
        now = datetime.now()
        activities = act.load_activity_log()
        activities.append({
            "timestamp": now.isoformat(),
            "date_str": now.strftime("%Y-%m-%d"),
            "time_str": now.strftime("%I:%M %p"),
            "time": now.strftime("%I:%M %p"),
            "name": name.value.strip(),
            "duration": duration.value,
            "intensity": intensity.value,
            "sensory": sensory.value,
        })
        act.save_activity_log(activities)
        flash(status, f"Logged {name.value.strip()}!")
        name.value = ""
        refresh_all()

    add.on_click(on_add)
    register(refresh)
    return w.VBox([w.HTML("<b>Today's activity</b>"), listing, w.HBox([name, duration]),
                   w.HBox([intensity, sensory, add]), status])


display(build_activity_section())

## 6. 📅 Schedule & Calendar
Pick a day to see its events. Today's medications, meals, and activities sync onto the calendar automatically. Deleted synced events stay deleted, just as in the app.

In [ ]:
def _event_day(event):
    return event.get("date") or str(event.get("start", ""))[:10]


def _event_time(event):
    if event.get("allDay") or "T" not in str(event.get("start", "")):
        return "All day"
    fmt = lambda s: datetime.fromisoformat(str(s)).strftime("%I:%M %p").lstrip("0")
    try:
        return f"{fmt(event['start'])} – {fmt(event['end'])}" if event.get("end") else fmt(event["start"])
    except ValueError:
        return str(event.get("start"))


def build_calendar_section():
    day = w.DatePicker(value=datetime.now().date(), description="Day")
    rows_box = w.VBox()
    status = status_label()

    def refresh():
        selected = (day.value or datetime.now().date()).strftime("%Y-%m-%d")
        events = sorted((e for e in cal.load_events() if _event_day(e) == selected),
                        key=lambda e: (not e.get("allDay", False), str(e.get("start", ""))))
        rows = []
        for e in events:
            chip = (f"<span style='display:inline-block;width:10px;height:10px;border-radius:2px;"
                    f"background:{esc(e.get('backgroundColor', '#6366F1'))};margin-right:6px'></span>")
            label = w.HTML(f"{chip}<b>{esc(_event_time(e))}</b> · {esc(e.get('title'))}",
                           layout=w.Layout(width="520px"))
            btn = w.Button(description="🗑️", tooltip="Delete event", layout=w.Layout(width="48px"))
            btn.on_click(make_delete(e.get("id"), e.get("title")))
            rows.append(w.HBox([label, btn]))
        rows_box.children = rows or [w.HTML("<i>No events on this day.</i>")]

    def make_delete(event_id, title):
        def handler(_):
            cal.save_events([e for e in cal.load_events() if e.get("id") != event_id])
            if event_id and event_id.startswith(cal.SYNCED_ID_PREFIXES):
                cal.dismiss_synced_event(event_id)
            flash(status, f"Deleted {title}.")
            refresh()
        return handler

    day.observe(lambda _: refresh(), names="value")

    title = w.Text(placeholder="e.g. Doctor Visit", description="Event")
    all_day = w.Checkbox(value=False, description="All day")
    start = w.TimePicker(value=dtime(9, 0), description="Start")
    end = w.TimePicker(value=dtime(10, 0), description="End")
    add = w.Button(description="➕ Add Event", button_style="primary")

    def on_add(_):
        if not title.value.strip():
            flash(status, "Please enter an event name.", ok=False)
            return
        date_part = (day.value or datetime.now().date()).strftime("%Y-%m-%d")
        if all_day.value:
            final_start = final_end = date_part
        else:
            s, e = start.value or dtime(9, 0), end.value or dtime(10, 0)
            if e <= s:
                flash(status, "End time must be after start time.", ok=False)
                return
            final_start, final_end = f"{date_part}T{s.strftime('%H:%M:%S')}", f"{date_part}T{e.strftime('%H:%M:%S')}"
        events = cal.load_events()
        events.append({
            "id": str(uuid.uuid4()),
            "title": title.value.strip(),
            "start": final_start,
            "end": final_end,
            "date": date_part,
            "allDay": all_day.value,
            "backgroundColor": "#6366F1",
            "borderColor": "#4338CA",
        })
        cal.save_events(events)
        flash(status, f"Added {title.value.strip()} on {date_part}.")
        title.value = ""
        refresh()

    add.on_click(on_add)
    sync_btn = w.Button(description="🔄 Sync Today's Logs")
    sync_btn.on_click(lambda _: (refresh_all(), flash(status, "Synced daily logs to schedule!")))

    register(refresh)
    return w.VBox([w.HBox([day, sync_btn]), rows_box, w.HTML("<b>Add an event on the selected day</b>"),
                   w.HBox([title, all_day]), w.HBox([start, end, add]), status])


display(build_calendar_section())

## 7. 🍱 Favorite Foods Registry
Low-demand go-to foods. The Decision Assistant suggests foods from this list.

In [ ]:
FOOD_COLUMNS = ["name", "category", "texture", "taste", "temp", "effort", "notes"]


def build_foods_section():
    search = w.Text(placeholder="Search by name, category, texture…", description="Filter",
                    continuous_update=False)
    table_out = w.Output()
    remove_pick = w.Dropdown(description="Remove")
    remove_btn = w.Button(description="🗑️ Remove", tooltip="Remove the selected food")
    status = status_label()

    def refresh():
        df = sfl.load_safe_foods()
        for col in FOOD_COLUMNS:
            if col not in df.columns:
                df[col] = ""
        if search.value.strip():
            q = search.value.strip()
            mask = pd.Series(False, index=df.index)
            for col in FOOD_COLUMNS:
                mask |= df[col].astype(str).str.contains(q, case=False, regex=False, na=False)
            df = df[mask]
        table_out.clear_output()
        with table_out:
            if df.empty:
                display(HTML("<i>No favorite foods found.</i>"))
            else:
                display(df[FOOD_COLUMNS].rename(columns=str.title).reset_index(drop=True))
        remove_pick.options = [(r["name"], r["id"]) for _, r in df.iterrows() if r.get("id")]

    def on_remove(_):
        if not remove_pick.value:
            return
        records = [r for r in sfl.load_safe_foods().to_dict(orient="records") if r.get("id") != remove_pick.value]
        sfl.save_safe_foods(records)
        flash(status, "Removed from registry.")
        refresh()

    search.observe(lambda _: refresh(), names="value")
    remove_btn.on_click(on_remove)

    name = w.Text(description="Food")
    category = w.Dropdown(options=["Breakfast", "Lunch", "Dinner", "Snack", "Drink", "Other"], description="Category")
    texture = w.Dropdown(options=["Soft", "Crunchy", "Smooth", "Chewy", "Liquid", "Gloopy", "Other"], description="Texture")
    taste = w.Dropdown(options=["Sweet", "Sour", "Salty", "Savory", "Other"], description="Taste")
    temp = w.Dropdown(options=["Cold", "Room Temperature", "Warm", "Hot", "Any"], description="Temp")
    effort = w.Dropdown(options=["None", "Low", "Medium", "High"], value="Low", description="Effort")
    notes = w.Text(placeholder="e.g. Microwave for 90s", description="Notes")
    add = w.Button(description="➕ Add Favorite Food", button_style="primary")

    def on_add(_):
        if not name.value.strip():
            flash(status, "Please enter a food name.", ok=False)
            return
        records = sfl.load_safe_foods().to_dict(orient="records")
        records.append({
            "id": f"food_{uuid.uuid4().hex[:8]}", "name": name.value.strip(), "category": category.value,
            "texture": texture.value, "taste": taste.value, "temp": temp.value,
            "effort": effort.value, "notes": notes.value.strip(),
        })
        sfl.save_safe_foods(records)
        flash(status, f"Added {name.value.strip()}!")
        name.value = notes.value = ""
        refresh()

    add.on_click(on_add)
    register(refresh)
    return w.VBox([search, table_out, w.HBox([remove_pick, remove_btn]), w.HTML("<b>Add a favorite food</b>"),
                   w.HBox([name, category, texture]), w.HBox([taste, temp, effort]), w.HBox([notes, add]), status])


display(build_foods_section())

## 8. 🛒 Shopping List
Changes are saved as soon as you tick a box.

In [ ]:
def build_shopping_section():
    items_box = w.VBox()
    status = status_label()

    def refresh():
        items = sfl.load_shopping_list()
        boxes = []
        for idx, entry in enumerate(items):
            cb = w.Checkbox(value=bool(entry.get("bought")), indent=False,
                            description=f"{entry.get('item')} ({entry.get('category', 'Other')})")
            cb.observe(make_toggle(idx), names="value")
            boxes.append(cb)
        items_box.children = boxes or [w.HTML("<i>Shopping list is empty.</i>")]

    def make_toggle(idx):
        def handler(change):
            items = sfl.load_shopping_list()
            if 0 <= idx < len(items):
                items[idx]["bought"] = change["new"]
                sfl.save_shopping_list(items)
        return handler

    item = w.Text(description="Item")
    category = w.Dropdown(options=["Produce", "Dairy", "Pantry", "Fridge", "Frozen", "Other"], description="Category")
    add = w.Button(description="Add Item", button_style="primary")
    clear = w.Button(description="Clear bought items")

    def on_add(_):
        if not item.value.strip():
            return
        items = sfl.load_shopping_list()
        items.append({"item": item.value.strip(), "category": category.value, "bought": False})
        sfl.save_shopping_list(items)
        flash(status, f"Added {item.value.strip()}.")
        item.value = ""
        refresh()

    def on_clear(_):
        items = sfl.load_shopping_list()
        sfl.save_shopping_list([i for i in items if not i.get("bought")])
        flash(status, f"Removed {sum(1 for i in items if i.get('bought'))} bought item(s).")
        refresh()

    add.on_click(on_add)
    clear.on_click(on_clear)
    register(refresh)
    return w.VBox([items_box, w.HBox([item, category, add]), clear, status])


display(build_shopping_section())

## 9. 🧠 Decision Assistant (local Llama 3.2)
Help deciding what to eat, breaking down a task, or de-escalating overload. It uses the same fine-tuned model and prompts as the app, served by your local Ollama.

In [ ]:
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.2-3b-ft")


def generate(system_prompt, user_prompt, temperature=0.2, timeout=120):
    """Calls Ollama with raw Llama 3 chat tokens, same as app.py. Returns (ok, text)."""
    system_prompt, user_prompt = str(system_prompt).strip(), str(user_prompt).strip()
    if len(user_prompt) < 2:
        return False, "Please enter a valid prompt before generating."
    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_prompt}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_prompt}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "raw": True,
        "stream": False,
        "keep_alive": "10m",  # keep the model warm between questions
        "options": {
            "temperature": float(temperature),
            "repeat_penalty": 1.18,
            "top_p": 0.9,
            "stop": ["<|eot_id|>", "<|eom_id|>", "<|start_header_id|>", "<|end_header_id|>"],
            "num_predict": 200,
        },
    }
    try:
        res = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=timeout)
    except requests.RequestException as exc:
        return False, f"Could not reach Ollama at {OLLAMA_URL}: {exc}"
    if res.status_code != 200:
        return False, f"HTTP {res.status_code}: {res.text}"
    text = str(res.json().get("response", "")).strip()
    return True, text or "No output returned."


def guess_meal_type(item_name):
    drink_words = ["water", "shake", "juice", "drink", "tea", "coffee", "smoothie", "liquid"]
    return "Drink" if any(word in item_name.lower() for word in drink_words) else "Meal"

In [ ]:
def build_assistant_section():
    mode = w.ToggleButtons(options=["Food Options", "Task Breakdown", "Chat Help with Overwhelm", "Custom Prompt"])

    energy = w.SelectionSlider(options=["None", "Low", "Medium", "High"], description="Energy",
                               style={"description_width": "initial"})
    texture = w.Dropdown(options=["Soft", "Crunchy", "Smooth", "Chewy", "Any", "Gloopy", "Liquid"], description="Texture")
    task = w.Text(placeholder="e.g. Cleaning the kitchen sink", description="Task",
                  layout=w.Layout(width="520px"))
    overwhelm = w.Textarea(value="Everything is too loud and I have too many things on my to-do list.",
                           layout=w.Layout(width="520px", height="70px"))
    custom_sys = w.Textarea(value="You are an empathetic, low-demand assistant. Help the user gently.",
                            description="System", layout=w.Layout(width="520px", height="60px"))
    custom_usr = w.Textarea(value="I need help deciding what to do next.", description="User",
                            layout=w.Layout(width="520px", height="60px"))
    custom_temp = w.FloatSlider(value=0.2, min=0.0, max=1.0, step=0.05, description="Temperature")

    forms = {
        "Food Options": w.VBox([w.HBox([energy, texture])]),
        "Task Breakdown": w.VBox([w.HTML("Enter a task causing executive paralysis; it will be split into ≤3 tiny steps."), task]),
        "Chat Help with Overwhelm": w.VBox([w.HTML("What is overwhelming you right now?"), overwhelm]),
        "Custom Prompt": w.VBox([custom_sys, custom_usr, custom_temp]),
    }
    form_box = w.VBox()
    go = w.Button(description="✨ Generate", button_style="primary")
    result = w.Output()
    quick_actions = w.HBox()
    status = status_label()

    def show_form(_=None):
        form_box.children = [forms[mode.value]]
        result.clear_output()
        quick_actions.children = []
        if mode.value == "Food Options":
            energy.value = energy_from_spoons(load_capacity()["spoons_count"])
            energy.description = f"Energy ({load_capacity()['spoons_count']} 🥄)"

    def build_prompts():
        if mode.value == "Food Options":
            foods = sfl.load_safe_foods()
            foods_json = foods.to_json(orient="records") if not foods.empty else "[]"
            return (f"You are a helpful peer assistant. Suggest 2 safe foods from this registry matching energy "
                    f"'{energy.value}' and texture '{texture.value}': {foods_json}",
                    f"Suggest 2 quick safe food ideas for Energy Level: {energy.value} and Texture Preference: {texture.value}.",
                    0.2)
        if mode.value == "Task Breakdown":
            return ("You are a low-demand executive dysfunction coach. Break tasks into 3 tiny, friction-free micro-steps.",
                    f"Task to break down: {task.value}" if task.value.strip() else "Task: Clean my desk space",
                    0.2)
        if mode.value == "Chat Help with Overwhelm":
            return ("You are a warm, low-demand support assistant. Provide short de-escalation guidance.",
                    overwhelm.value, 0.3)
        return custom_sys.value, custom_usr.value, custom_temp.value

    def make_logger(item_name):
        def handler(_):
            meal.log_meal_entry(item_name, guess_meal_type(item_name))
            flash(status, f"Logged {item_name} to today's meals!")
            refresh_all()
        return handler

    def on_generate(_):
        go.disabled = True
        go.description = "⏳ Generating…"
        quick_actions.children = []
        result.clear_output()
        try:
            ok, text = generate(*build_prompts())
        finally:
            go.disabled = False
            go.description = "✨ Generate"
        with result:
            if ok:
                display(Markdown(f"**Assistant Response:**\n\n{text}"))
            else:
                display(HTML(f"<span style='color:#b91c1c'>{esc(text)}</span>"))
        if ok and mode.value == "Food Options":
            matches = re.findall(r"\d+\.\s*\*\*([^*]+)\*\*", text) or re.findall(r"\*\*([^*]+)\*\*", text)
            buttons = []
            for item in matches[:2]:
                b = w.Button(description=f"🍱 Log {item.strip()}"[:40], tooltip=item.strip(),
                             layout=w.Layout(width="auto"))
                b.on_click(make_logger(item.strip()))
                buttons.append(b)
            quick_actions.children = buttons

    mode.observe(show_form, names="value")
    go.on_click(on_generate)
    show_form()
    return w.VBox([mode, form_box, go, result, quick_actions, status])


display(build_assistant_section())

## 10. All-in-one Dashboard
Every section in one tabbed view. Run this cell on its own to use the notebook like the app. All the sections above stay in sync with it.

In [ ]:
def build_dashboard():
    sections = [
        ("Capacity", build_capacity_section),
        ("💊 Meds", build_medication_section),
        ("🍽️ Meals", build_meal_section),
        ("🧘 Activity", build_activity_section),
        ("📅 Calendar", build_calendar_section),
        ("🍱 Foods", build_foods_section),
        ("🛒 Shopping", build_shopping_section),
        ("🧠 Assistant", build_assistant_section),
    ]
    tabs = w.Tab(children=[build() for _, build in sections])
    tabs.titles = [title for title, _ in sections]
    return tabs


cal.sync_daily_logs_to_calendar()
display(build_dashboard())